# Notebook 03 – Weighted Overlay Analysis (Per 10-Year Period)

Computes a weighted overlay for each 10-year period (1990–2020) using AHP weights from `data/interim/AHP_Vægtninger.xlsx` and reclassified rasters from `data/processed/tsc_reclass/`.

Outputs per period saved to `results/metrics/overlay_output/`:
- `weighted_overlay_{period}.tif`
- `weighted_overlay_clusters_{period}.gpkg` / `.geojson`
- `weighted_overlay_statistics_{period}.csv`
- `weighted_overlay_map_{period}.html`

## Imports & paths

In [1]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import fiona
import folium
import openpyxl
from pathlib import Path
from rasterstats import zonal_stats
from shapely.geometry import shape as geom_from_shape

REPO_ROOT     = Path('..').resolve()
DATA_DIR      = REPO_ROOT / 'data'
RAW_DIR       = DATA_DIR / 'raw'
RESULTS_DIR   = REPO_ROOT / 'results'

RECLASS_FOLDER = DATA_DIR / 'processed' / 'tsc_reclass'
OUTPUT_FOLDER  = RESULTS_DIR / 'metrics' / 'overlay_output'
CLUSTERS_SHP   = RAW_DIR / 'nabolag_inkl_y_kom_shapefile' / 'cluster_outer_v1.shp'
WEIGHTS_EXCEL  = DATA_DIR / 'interim' / 'AHP_Vægtninger.xlsx'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f'Repo root:      {REPO_ROOT}')
print(f'Reclass folder: {RECLASS_FOLDER}')
print(f'Output folder:  {OUTPUT_FOLDER}')
print(f'Clusters SHP:   {CLUSTERS_SHP.exists()}')
print(f'Weights Excel:  {WEIGHTS_EXCEL.exists()}')

Repo root:      C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model
Reclass folder: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_reclass
Output folder:  C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\results\metrics\overlay_output
Clusters SHP:   True
Weights Excel:  True


## Load AHP weights from Excel

In [2]:
wb = openpyxl.load_workbook(WEIGHTS_EXCEL)
ws = wb.active

weight_data = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0] is not None:
        weight_data.append((str(row[0]), float(row[1])))

weights_dict_raw = {var: weight for var, weight in weight_data}

print(f'Weights loaded: {len(weights_dict_raw)} variables')
for var, w in sorted(weights_dict_raw.items(), key=lambda x: x[1], reverse=True):
    print(f'  {var:30s}: {w:.4f}')

Weights loaded: 20 variables
  disp_inc                      : 12.0500
  public_housing                : 9.7167
  mean_sqm                      : 8.8833
  mean_price                    : 8.7833
  mig_out                       : 7.1167
  unemp                         : 6.7833
  lvu                           : 6.3833
  emp                           : 6.1667
  mig_in                        : 5.8000
  ool                           : 4.9500
  age_26_40                     : 3.7333
  grund                         : 3.4667
  gym_erhv                      : 2.7000
  crime_main_y                  : 2.3500
  age_41_55                     : 2.2333
  PUB                           : 2.2000
  PMB                           : 2.0667
  age_18_25                     : 1.7333
  age_56_69                     : 1.5000
  EMUB                          : 1.4167


## Exclude layers & normalize weights

Excluded: `qol`, `mig_net`, `counts` (same as OLD_03)

In [3]:
excluded = ['qol', 'mig_net', 'counts']

weight_dict = {k: v for k, v in weights_dict_raw.items() if k not in excluded}

total_weight = sum(weight_dict.values())
normalized_weights = {k: v / total_weight for k, v in weight_dict.items()}

print(f'Variables after exclusion: {len(weight_dict)} (excluded: {excluded})')
print(f'Sum of normalized weights: {sum(normalized_weights.values()):.6f}')
print('\nNormalized weights (sorted):')
for var, w in sorted(normalized_weights.items(), key=lambda x: x[1], reverse=True):
    print(f'  {var:30s}: {w:.4f}')

Variables after exclusion: 20 (excluded: ['qol', 'mig_net', 'counts'])
Sum of normalized weights: 1.000000

Normalized weights (sorted):
  disp_inc                      : 0.1205
  public_housing                : 0.0971
  mean_sqm                      : 0.0888
  mean_price                    : 0.0878
  mig_out                       : 0.0711
  unemp                         : 0.0678
  lvu                           : 0.0638
  emp                           : 0.0616
  mig_in                        : 0.0580
  ool                           : 0.0495
  age_26_40                     : 0.0373
  grund                         : 0.0347
  gym_erhv                      : 0.0270
  crime_main_y                  : 0.0235
  age_41_55                     : 0.0223
  PUB                           : 0.0220
  PMB                           : 0.0207
  age_18_25                     : 0.0173
  age_56_69                     : 0.0150
  EMUB                          : 0.0142


## Discover rasters grouped by period

Filename pattern: `reclass_{var}_{start}_{end}_tsc.tif`

In [4]:
PERIODS = ['1990_2000', '2000_2010', '2010_2020']

rasters_by_period = {p: {} for p in PERIODS}

for fname in os.listdir(RECLASS_FOLDER):
    if not fname.endswith('.tif'):
        continue
    for period in PERIODS:
        suffix = f'_{period}_tsc.tif'
        if fname.endswith(suffix):
            var = fname[len('reclass_'):-len(suffix)]
            rasters_by_period[period][var] = RECLASS_FOLDER / fname
            break

print('Rasters discovered per period:')
for period, rasters in rasters_by_period.items():
    start, end = period.split('_')
    print(f'  {start}–{end}: {len(rasters)} rasters')

Rasters discovered per period:
  1990–2000: 0 rasters
  2000–2010: 0 rasters
  2010–2020: 0 rasters


## Load clusters shapefile

In [5]:
geometries = []
properties_list = []

with fiona.open(CLUSTERS_SHP) as src:
    shp_crs = src.crs
    for feature in src:
        geometries.append(geom_from_shape(feature['geometry']))
        properties_list.append(feature['properties'])

clusters_gdf = gpd.GeoDataFrame(properties_list, geometry=geometries, crs=shp_crs)

print(f'Clusters loaded: {len(clusters_gdf)} features')
print(f'CRS: {clusters_gdf.crs}')
print(f'Columns: {list(clusters_gdf.columns)}')


def get_color(value):
    """5-tier red→green color ramp based on normalized overlay value."""
    if pd.isna(value) or value < 0.2:
        return '#d73027'
    elif value < 0.4:
        return '#fc8d59'
    elif value < 0.6:
        return '#fee090'
    elif value < 0.8:
        return '#91bfdb'
    else:
        return '#1a9850'

Clusters loaded: 2232 features
CRS: PROJCS["ETRS89 / UTM zone 32N",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","25832"]]
Columns: ['area', 'cluster_id', 'fid', 'id_munic', 'munic_code', 'geometry']


## Per-period weighted overlay loop

In [6]:
summary_rows = []

for period in PERIODS:
    start, end = period.split('_')
    print(f"\n{'='*60}")
    print(f'Processing period: {start}–{end}')
    print(f"{'='*60}")

    period_rasters = rasters_by_period.get(period, {})
    if not period_rasters:
        print(f'  No rasters found for {period}, skipping.')
        continue

    # Match normalized weights to rasters available for this period; re-normalize
    matched = {k: normalized_weights[k] for k in normalized_weights if k in period_rasters}
    if not matched:
        print(f'  No weight matches for {period}, skipping.')
        continue

    total_w = sum(matched.values())
    matched_norm = {k: v / total_w for k, v in matched.items()}

    print(f'  Rasters found:   {len(period_rasters)}')
    print(f'  Weights matched: {len(matched_norm)}')

    # --- Grid properties from first raster ---
    first_path = list(period_rasters.values())[0]
    with rasterio.open(first_path) as src:
        raster_crs = src.crs
        raster_transform = src.transform
        height = src.height
        width = src.width

    # --- Weighted overlay (per-pixel weight normalisation) ---
    # Accumulate weighted sum AND the sum of weights that actually had valid data
    # per pixel. Dividing at the end ensures boundary pixels with missing variables
    # are on the same scale as fully-covered interior pixels, eliminating spurious
    # within-cluster variation caused by edge nodata in individual rasters.
    overlay    = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    for var, weight in sorted(matched_norm.items(), key=lambda x: x[1], reverse=True):
        with rasterio.open(period_rasters[var]) as src:
            data = src.read(1).astype(np.float32)
        valid = data > 0
        overlay    += np.where(valid, data * weight, 0.0)
        weight_sum += np.where(valid, weight,        0.0)

    # Normalise: weighted mean using only contributing variables per pixel
    overlay = np.where(weight_sum > 0, overlay / weight_sum, np.nan)

    print(f'  Overlay range: {np.nanmin(overlay):.4f} – {np.nanmax(overlay):.4f}')
    print(f'  Overlay mean:  {np.nanmean(overlay):.4f}')

    # --- Save overlay raster (NaN → 0) ---
    overlay_save = np.where(np.isnan(overlay), 0, overlay).astype(np.float32)
    overlay_path = OUTPUT_FOLDER / f'weighted_overlay_{period}.tif'

    with rasterio.open(
        overlay_path, 'w',
        driver='GTiff', height=height, width=width,
        count=1, dtype=np.float32,
        crs=raster_crs, transform=raster_transform
    ) as dst:
        dst.write(overlay_save, 1)
    print(f'  Saved raster: {overlay_path.name}')

    # --- Zonal statistics ---
    period_gdf = clusters_gdf.copy()
    if period_gdf.crs != raster_crs:
        period_gdf = period_gdf.to_crs(raster_crs)

    stats_list = zonal_stats(
        period_gdf.geometry,
        str(overlay_path),
        affine=raster_transform,
        stats=['mean', 'count', 'std', 'min', 'max'],
        nodata=0,
        all_touched=False
    )
    stats_df = pd.DataFrame(stats_list)
    stats_df['range'] = stats_df['max'] - stats_df['min']

    result_gdf = period_gdf.copy()
    for col in stats_df.columns:
        result_gdf[col] = stats_df[col].values

    # --- Save vector outputs ---
    gpkg_path    = OUTPUT_FOLDER / f'weighted_overlay_clusters_{period}.gpkg'
    geojson_path = OUTPUT_FOLDER / f'weighted_overlay_clusters_{period}.geojson'
    csv_path     = OUTPUT_FOLDER / f'weighted_overlay_statistics_{period}.csv'

    # Drop 'fid' column if present — conflicts with GeoPackage/GeoJSON auto-FID
    fid_cols = [c for c in result_gdf.columns if c.lower() == 'fid']
    save_gdf = result_gdf.drop(columns=fid_cols) if fid_cols else result_gdf

    save_gdf.to_file(str(gpkg_path), driver='GPKG')
    save_gdf.to_file(str(geojson_path), driver='GeoJSON')
    save_gdf.drop(columns='geometry').to_csv(str(csv_path), index=False)
    print(f'  Saved: {gpkg_path.name}, {geojson_path.name}, {csv_path.name}')

    # --- Folium map ---
    map_gdf = result_gdf.to_crs('EPSG:4326')
    center_lat = map_gdf.geometry.centroid.y.mean()
    center_lon = map_gdf.geometry.centroid.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')

    min_val = float(map_gdf['mean'].min())
    max_val = float(map_gdf['mean'].max())
    denom = (max_val - min_val) if max_val > min_val else 1.0
    map_gdf = map_gdf.copy()
    map_gdf['normalized_mean'] = (map_gdf['mean'] - min_val) / denom

    for idx, row in map_gdf.iterrows():
        color = get_color(row['normalized_mean'])
        count_val = int(row['count']) if not pd.isna(row['count']) else 'N/A'
        popup_text = (
            f"<b>Period: {start}–{end}</b><br><hr>"
            f"Mean: {row['mean']:.4f}<br>"
            f"Std: {row['std']:.4f}<br>"
            f"Count: {count_val}<br>"
            f"Range: {row['range']:.4f}<br>"
            f"Min: {row['min']:.4f} &nbsp; Max: {row['max']:.4f}"
        )
        folium.GeoJson(
            data=row.geometry.__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c, 'color': 'black',
                'weight': 1.5, 'opacity': 0.9, 'fillOpacity': 0.7
            },
            popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)

    legend_html = f"""
    <div style="position:fixed;bottom:50px;right:50px;width:230px;
                background:white;border:2px solid grey;z-index:9999;
                font-size:13px;padding:10px;border-radius:5px;">
      <p style="margin:0;font-weight:bold;text-align:center;">Period {start}–{end}<br>Mean Overlay Value</p>
      <hr style="margin:5px 0;">
      <p style="margin:3px 0;"><i style="background:#d73027;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very Low (0–20%)</p>
      <p style="margin:3px 0;"><i style="background:#fc8d59;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Low (20–40%)</p>
      <p style="margin:3px 0;"><i style="background:#fee090;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Medium (40–60%)</p>
      <p style="margin:3px 0;"><i style="background:#91bfdb;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>High (60–80%)</p>
      <p style="margin:3px 0;"><i style="background:#1a9850;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very High (80–100%)</p>
      <hr style="margin:8px 0;">
      <p style="margin:3px 0;font-size:11px;"><b>Min:</b> {min_val:.4f}</p>
      <p style="margin:3px 0;font-size:11px;"><b>Max:</b> {max_val:.4f}</p>
      <p style="margin:3px 0;font-size:11px;"><b>Clusters:</b> {len(map_gdf)}</p>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    map_path = OUTPUT_FOLDER / f'weighted_overlay_map_{period}.html'
    m.save(str(map_path))
    print(f'  Saved map: {map_path.name}')

    std_valid  = stats_df['std'].dropna()
    std_nonzero = int((std_valid > 0).sum())
    total_clusters = int(std_valid.count())

    summary_rows.append({
        'period': f'{start}–{end}',
        'rasters_matched': len(matched_norm),
        'clusters_with_data': int(stats_df['count'].notna().sum()),
        'mean_overlay': round(float(np.nanmean(overlay_save)), 4),
        'max_overlay': round(float(np.nanmax(overlay_save)), 4),
        'std_nonzero': std_nonzero,
        'total_clusters': total_clusters,
    })

print(f"\n{'='*60}")
print('All periods processed.')



Processing period: 1990–2000
  No rasters found for 1990_2000, skipping.

Processing period: 2000–2010
  No rasters found for 2000_2010, skipping.

Processing period: 2010–2020
  No rasters found for 2010_2020, skipping.

All periods processed.


## Summary

In [7]:
summary_df = pd.DataFrame(summary_rows)
print('Weighted Overlay Summary')
print('=' * 60)
print(summary_df.to_string(index=False))

print('\nClusters with std > 0 per period:')
for _, row in summary_df.iterrows():
    print(f"  {row['period']}: {row['std_nonzero']}/{row['total_clusters']} std not zero")

outputs = sorted(OUTPUT_FOLDER.glob('weighted_overlay_*.tif'))
print(f'\nOutput rasters in {OUTPUT_FOLDER.relative_to(REPO_ROOT)}:')
for f in outputs:
    print(f'  {f.name}')

Weighted Overlay Summary
Empty DataFrame
Columns: []
Index: []

Clusters with std > 0 per period:

Output rasters in results\metrics\overlay_output:


## Full-period weighted overlay (1990–2020)

Single weighted overlay using all `reclass_*_1990_2020_tsc.tif` rasters with the same AHP weights.


In [8]:
FULL_PERIOD = '1990_2020'
suffix_full = f'_{FULL_PERIOD}_tsc.tif'

# Discover full-period rasters
full_rasters = {}
for fname in os.listdir(RECLASS_FOLDER):
    if fname.endswith(suffix_full):
        var = fname[len('reclass_'):-len(suffix_full)]
        full_rasters[var] = RECLASS_FOLDER / fname

print(f'Full-period rasters found: {len(full_rasters)}')
for v in sorted(full_rasters):
    print(f'  {v}')

# Match and re-normalize weights
matched_full = {k: normalized_weights[k] for k in normalized_weights if k in full_rasters}
total_w_full = sum(matched_full.values())
matched_full_norm = {k: v / total_w_full for k, v in matched_full.items()}

print(f'\nWeights matched: {len(matched_full_norm)}  (sum={sum(matched_full_norm.values()):.6f})')

# Grid properties from first raster
first_path_full = list(full_rasters.values())[0]
with rasterio.open(first_path_full) as src:
    raster_crs_full = src.crs
    raster_transform_full = src.transform
    height_full = src.height
    width_full  = src.width

# Weighted overlay
overlay_full    = np.zeros((height_full, width_full), dtype=np.float32)
weight_sum_full = np.zeros((height_full, width_full), dtype=np.float32)

for var, weight in sorted(matched_full_norm.items(), key=lambda x: x[1], reverse=True):
    with rasterio.open(full_rasters[var]) as src:
        data = src.read(1).astype(np.float32)
    valid = data > 0
    overlay_full    += np.where(valid, data * weight, 0.0)
    weight_sum_full += np.where(valid, weight,        0.0)

overlay_full = np.where(weight_sum_full > 0, overlay_full / weight_sum_full, np.nan)

print(f'\nOverlay range: {np.nanmin(overlay_full):.4f} – {np.nanmax(overlay_full):.4f}')
print(f'Overlay mean:  {np.nanmean(overlay_full):.4f}')

# Save raster
overlay_full_save = np.where(np.isnan(overlay_full), 0, overlay_full).astype(np.float32)
overlay_full_path = OUTPUT_FOLDER / f'weighted_overlay_{FULL_PERIOD}.tif'

with rasterio.open(
    overlay_full_path, 'w',
    driver='GTiff', height=height_full, width=width_full,
    count=1, dtype=np.float32,
    crs=raster_crs_full, transform=raster_transform_full
) as dst:
    dst.write(overlay_full_save, 1)
print(f'Saved raster: {overlay_full_path.name}')

# Zonal statistics
gdf_full = clusters_gdf.copy()
if gdf_full.crs != raster_crs_full:
    gdf_full = gdf_full.to_crs(raster_crs_full)

stats_full = zonal_stats(
    gdf_full.geometry,
    str(overlay_full_path),
    affine=raster_transform_full,
    stats=['mean', 'count', 'std', 'min', 'max'],
    nodata=0,
    all_touched=False
)
stats_full_df = pd.DataFrame(stats_full)
stats_full_df['range'] = stats_full_df['max'] - stats_full_df['min']

result_gdf_full = gdf_full.copy()
for col in stats_full_df.columns:
    result_gdf_full[col] = stats_full_df[col].values

# Save vector outputs
fid_cols = [c for c in result_gdf_full.columns if c.lower() == 'fid']
save_gdf_full = result_gdf_full.drop(columns=fid_cols) if fid_cols else result_gdf_full

save_gdf_full.to_file(str(OUTPUT_FOLDER / f'weighted_overlay_clusters_{FULL_PERIOD}.gpkg'), driver='GPKG')
save_gdf_full.to_file(str(OUTPUT_FOLDER / f'weighted_overlay_clusters_{FULL_PERIOD}.geojson'), driver='GeoJSON')
save_gdf_full.drop(columns='geometry').to_csv(str(OUTPUT_FOLDER / f'weighted_overlay_statistics_{FULL_PERIOD}.csv'), index=False)
print(f'Saved: weighted_overlay_clusters_{FULL_PERIOD}.gpkg / .geojson / .csv')

# Folium map
map_gdf_full = result_gdf_full.to_crs('EPSG:4326')
center_lat = map_gdf_full.geometry.centroid.y.mean()
center_lon = map_gdf_full.geometry.centroid.x.mean()

m_full = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='OpenStreetMap')

min_val = float(map_gdf_full['mean'].min())
max_val = float(map_gdf_full['mean'].max())
denom = (max_val - min_val) if max_val > min_val else 1.0
map_gdf_full = map_gdf_full.copy()
map_gdf_full['normalized_mean'] = (map_gdf_full['mean'] - min_val) / denom

for idx, row in map_gdf_full.iterrows():
    color = get_color(row['normalized_mean'])
    count_val = int(row['count']) if not pd.isna(row['count']) else 'N/A'
    popup_text = (
        f"<b>Period: 1990–2020</b><br><hr>"
        f"Mean: {row['mean']:.4f}<br>"
        f"Std: {row['std']:.4f}<br>"
        f"Count: {count_val}<br>"
        f"Range: {row['range']:.4f}<br>"
        f"Min: {row['min']:.4f} &nbsp; Max: {row['max']:.4f}"
    )
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, c=color: {
            'fillColor': c, 'color': 'black',
            'weight': 1.5, 'opacity': 0.9, 'fillOpacity': 0.7
        },
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m_full)

legend_html_full = f"""
<div style="position:fixed;bottom:50px;right:50px;width:230px;
            background:white;border:2px solid grey;z-index:9999;
            font-size:13px;padding:10px;border-radius:5px;">
  <p style="margin:0;font-weight:bold;text-align:center;">1990–2020 Full Period<br>Mean Overlay Value</p>
  <hr style="margin:5px 0;">
  <p style="margin:3px 0;"><i style="background:#d73027;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very Low (0–20%)</p>
  <p style="margin:3px 0;"><i style="background:#fc8d59;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Low (20–40%)</p>
  <p style="margin:3px 0;"><i style="background:#fee090;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Medium (40–60%)</p>
  <p style="margin:3px 0;"><i style="background:#91bfdb;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>High (60–80%)</p>
  <p style="margin:3px 0;"><i style="background:#1a9850;width:18px;height:18px;display:inline-block;margin-right:5px;"></i>Very High (80–100%)</p>
  <hr style="margin:8px 0;">
  <p style="margin:3px 0;font-size:11px;"><b>Min:</b> {min_val:.4f}</p>
  <p style="margin:3px 0;font-size:11px;"><b>Max:</b> {max_val:.4f}</p>
  <p style="margin:3px 0;font-size:11px;"><b>Clusters:</b> {len(map_gdf_full)}</p>
</div>"""
m_full.get_root().html.add_child(folium.Element(legend_html_full))

map_full_path = OUTPUT_FOLDER / f'weighted_overlay_map_{FULL_PERIOD}.html'
m_full.save(str(map_full_path))
print(f'Saved map: {map_full_path.name}')


Full-period rasters found: 20
  EMUB
  PMB
  PUB
  age_18_25
  age_26_40
  age_41_55
  age_56_69
  crime_main_y
  disp_inc
  emp
  grund
  gym_erhv
  lvu
  mean_price
  mean_sqm
  mig_in
  mig_out
  ool
  public_housing
  unemp

Weights matched: 20  (sum=1.000000)

Overlay range: 1.0000 – 4.8825
Overlay mean:  3.2151
Saved raster: weighted_overlay_1990_2020.tif


C:\Users\jonas\AppData\Local\Temp\ipykernel_21284\2023683676.py:41: RuntimeWarning: invalid value encountered in divide
  overlay_full = np.where(weight_sum_full > 0, overlay_full / weight_sum_full, np.nan)


Saved: weighted_overlay_clusters_1990_2020.gpkg / .geojson / .csv


C:\Users\jonas\AppData\Local\Temp\ipykernel_21284\2023683676.py:90: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = map_gdf_full.geometry.centroid.y.mean()
C:\Users\jonas\AppData\Local\Temp\ipykernel_21284\2023683676.py:91: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = map_gdf_full.geometry.centroid.x.mean()


Saved map: weighted_overlay_map_1990_2020.html
